In [12]:
print("yo")
print("oh no")

yo
oh no


## Utils

In [13]:
!pip install imbalanced-learn

In [14]:
# Source - https://stackoverflow.com/a
# Posted by Afshin Amiri, modified by community. See post 'Timeline' for change history
# Retrieved 2025-11-17, License - CC BY-SA 4.0

# ---- RUN THIS TO UNZIP FOLDER ----
#import zipfile as zf
#files = zf.ZipFile("datasets.zip", 'r')
#files.extractall('./')
#files.close()


## Abalone-17_vs_7-8-9-10 dataset

In [15]:
import numpy as np
import os.path
import pandas as pd
from sklearn.preprocessing import StandardScaler

### Prepare data

In [16]:
file_name = "./datasets/abalone-17_vs_7-8-9-10/abalone-17_vs_7-8-9-10.dat"
#if os.path.exists(file_name):
#    print("exists")

df = pd.read_csv(
    file_name,
    comment='@',
    header=None,
    sep=','
)

columns = [
    "Sex", "Length", "Diameter", "Height",
    "Whole_weight", "Shucked_weight",
    "Viscera_weight", "Shell_weight",
    "Class"
]

df.columns = columns

df = pd.get_dummies(df, columns=["Sex"], dtype=int)

# print(df["Class"].unique())
df["Class"] = df["Class"].map({" negative": 0, " positive": 1})

X = df.drop("Class", axis=1).values
y = df["Class"].values


# print(type(df))
# print(df)
# #print(df[0])

# print('')
# print('X (Note: ):')
# print(X)

# print('')
# print('y (Note: ):')
# print(y)


df

#folds = load_data(file_name, type_data="imbalanced", raw=True)

,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Class,Sex_F,Sex_I,Sex_M
0,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,0,0,0,1
1,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,0,0,1,0
2,0.355,0.280,0.085,0.2905,0.0950,0.0395,0.115,0,0,1,0
3,0.365,0.295,0.080,0.2555,0.0970,0.0430,0.100,0,0,0,1
4,0.390,0.295,0.095,0.2030,0.0875,0.0450,0.075,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...
2333,0.570,0.440,0.140,0.9535,0.3785,0.2010,0.305,1,1,0,0
2334,0.585,0.455,0.125,1.0270,0.3910,0.2120,0.250,1,0,0,1
2335,0.620,0.485,0.220,1.5110,0.5095,0.2840,0.510,1,1,0,0
2336,0.635,0.505,0.185,1.3035,0.5010,0.2950,0.410,1,1,0,0


### Pipeline for imbalanced learning

In [17]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import ADASYN

from sklearn.neural_network import MLPClassifier

# pipeline = ImbPipeline(
#     steps=[
#         ("smote", SMOTE(random_state=0)),
#         ("mlp", MLPClassifier(
#             (5,10,5), 
#             max_iter=2000, 
#             solver='adam', 
#             learning_rate_init=0.001))
#     ]
# )

pipeline = MLPClassifier(
           (5,10,5), 
           max_iter=2000, 
           solver='adam', 
           learning_rate_init=0.001)

#pipeline = ImbPipeline(
#    steps=[
#        ("adasyn", ADASYN(random_state=0)),
#        ("mlp", MLPClassifier(
#            (5,10,5), 
#            max_iter=2000, 
#            solver='adam', 
#            learning_rate_init=0.001))
#    ]
#)



### Data cleaning

In [18]:
X = df.drop(columns=["Class", "Sex_F", "Sex_I", "Sex_M"], errors='ignore').values
y = df["Class"].values

count_class = df["Class"].value_counts()

imb_count = df["Class"].value_counts(normalize=True) * 100
imb_count

imb_ratio = count_class.max() / count_class.min()
imb_ratio

39.310344827586206

### Test

In [19]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score

valid_precisions = []
valid_recalls = []
valid_f1s = []

runs_target = 100
runs_cur = 0
seed = 0

while runs_cur < runs_target:
    # Random Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, 
        test_size=0.3, 
        stratify=None, 
        random_state=seed
    )
    
    n_minority = sum(y_test == 1)
    
    if n_minority >= 10:
        # Scale
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        
        # Train
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        
        valid_precisions.append(precision_score(y_test, y_pred, pos_label=1, zero_division=0))
        valid_recalls.append(recall_score(y_test, y_pred, pos_label=1, zero_division=0))
        valid_f1s.append(f1_score(y_test, y_pred, pos_label=1, zero_division=0))
        
        runs_cur += 1
        
        if runs_cur % 10 == 0:
            print(f"Run {runs_cur}/{runs_target} completed...")

    seed += 1


print(f"Seeds checked: {seed}")
print(f"Average Precision: {np.mean(valid_precisions):.4f}")
print(f"Average Recall:    {np.mean(valid_recalls):.4f}")
print(f"Average F1:        {np.mean(valid_f1s):.4f}")

Run 10/100 completed...
Run 20/100 completed...
Run 30/100 completed...
Run 40/100 completed...
Run 50/100 completed...
Run 60/100 completed...
Run 70/100 completed...
Run 80/100 completed...
Run 90/100 completed...
Run 100/100 completed...
Seeds checked: 100
Average Precision: 0.4681
Average Recall:    0.1458
Average F1:        0.2069
